In [2]:
import numpy as np
import pandas as pd

from datetime import datetime, timedelta
import os
from time import sleep
import json
import re

import warnings
warnings.filterwarnings("ignore")

import pickle as pkl

from pprint import pprint

from libraries.preprocess import preprocess

In [3]:
%%time

# Lee todos los archivos de la carpeta de "json_searches_spain"
# Crea una lista con los json

FOLDER_PATH = "json_searches_spain/" 

files = os.listdir(FOLDER_PATH)
jobs_results = list()
search_date = list()

for file in files:
    
    with open(file = f"{FOLDER_PATH}{file}", mode = "r") as f:
        x = json.load(f)
        
    if "jobs_results" in x:
        jobs_results.extend(x["jobs_results"])
        
        if "search_metadata" in x:
            date = x["search_metadata"]["created_at"]
            date = datetime.strptime(x["search_metadata"]["created_at"], "%Y-%m-%d %H:%M:%S UTC").date()
            
            for i in range(len(x["jobs_results"])):
                search_date.append(date)
    
print(len(jobs_results))
print(len(search_date))

86329
86329
CPU times: total: 4.12 s
Wall time: 6.25 s


In [4]:
df = pd.json_normalize(jobs_results)

del jobs_results

df["date_posted"] = search_date

df.shape

(86329, 15)

In [5]:
df = df.drop_duplicates("job_id").reset_index(drop = True)

df.shape

(86168, 15)

In [6]:
df = df.drop_duplicates("description").reset_index(drop = True)

df.shape

(22194, 15)

In [7]:
%%time

# Eliminamos duplicados
df = df.drop_duplicates("job_id").reset_index(drop = True) 

# Limpiamos "via"
for i in range(len(df["via"])):
    
    # Debido a valores NoneType, usamos try/except
    
    try:
        df.loc[i, "via"] = preprocess.clean_source(df.loc[i, "via"]) # Cada fila
        
    except:
        df.loc[i, "via"] = np.nan
    
# Limpiamos "location"
df["location"] = df["location"].apply(lambda x : preprocess.clean_location(x))

# Limpiamos "contract_type"
df["detected_extensions.schedule_type"] = df["detected_extensions.schedule_type"].apply(lambda x : preprocess.clean_contract_type(x))

# Limpiamos "created_date"
df["detected_extensions.posted_at"] = df["detected_extensions.posted_at"].apply(lambda x : preprocess.transform_date(x))

df["date_posted"] = [preprocess.get_date(x, y) for x, y in df[["date_posted", "detected_extensions.posted_at"]].values]

tech_skills = list()

# Creamos "tech_skills"
for i in range(df["description"].shape[0]):
    
    if not pd.isna(df.loc[i, "description"]):
    
    # try:
        tech_skills.append(preprocess.get_skills(df.loc[i, "description"])) # Cada fila
        
    else:
        tech_skills.append(np.nan)
        
df["tech_skills"] = tech_skills

CPU times: total: 4min 1s
Wall time: 17min 4s


In [8]:
df

,title,company_name,location,via,description,job_highlights,related_links,extensions,job_id,detected_extensions.posted_at,detected_extensions.schedule_type,thumbnail,detected_extensions.work_from_home,detected_extensions.salary,date_posted,tech_skills
0,Desarrollador de Software,THE WISE SEEKER S.L,Spain,Infoempleo,- Desarrollar soluciones de software asegurand...,[{'items': ['- Desarrollar soluciones de softw...,"[{'link': 'http://www.thewiseseeker.com/', 'te...","[17 hours ago, Full-time, No degree mentioned]",eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIFNvZn...,0 days 17:00:00,Full-time,NaN,NaN,NaN,2024-04-17,[]
1,Desarrollador de Software,WITRON Logistik + Informatik GmbH,"Madrid, Spain",Glassdoor,Als Teil der WITRON Gruppe plant und realisier...,[{'items': ['Als Teil der WITRON Gruppe plant ...,"[{'link': 'http://www.witron.de/', 'text': 'wi...","[28 days ago, Full-time, No degree mentioned]",eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIFNvZn...,28 days 00:00:00,Full-time,https://encrypted-tbn0.gstatic.com/images?q=tb...,NaN,NaN,2024-03-20,[SQL]
2,Desarrollador de software,CBT,"Getxo, Spain",LinkedIn,CBT es una empresa del Grupo Innovalia dedicad...,[{'items': ['CBT es una empresa del Grupo Inno...,[{'link': 'https://www.google.com/search?sca_e...,"[2 days ago, Full-time, No degree mentioned]",eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIHNvZn...,2 days 00:00:00,Full-time,https://encrypted-tbn0.gstatic.com/images?q=tb...,NaN,NaN,2024-04-15,"[Angular, Android, iPhone]"
3,Desarrollador/a de software en C/C+,SEGULA Technologies,"Barcelona, Spain",LinkedIn,Localidad : Barcelona\n\nProvincia : Barcelona...,[{'items': ['Localidad : Barcelona Provincia ...,[{'link': 'http://www.segulatechnologies.com/'...,"[2 days ago, Full-time, No degree mentioned]",eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yL2EgZGUgc2...,2 days 00:00:00,Full-time,https://encrypted-tbn0.gstatic.com/images?q=tb...,NaN,NaN,2024-04-15,"[Jira, REST]"
4,Desarrollador de PHP,Proconsi,"León, Spain",LinkedIn,𝐏𝐫𝐨𝐠𝐫𝐚𝐦𝐚𝐝𝐨𝐫𝐞𝐬/𝐚𝐬 𝐏𝐇𝐏 𝐋𝐀𝐑𝐀𝐕𝐄𝐋 👩💻 👨💻\n\n¿𝐏𝐫𝐨𝐠𝐫𝐚𝐦...,[{'items': ['𝐏𝐫𝐨𝐠𝐫𝐚𝐦𝐚𝐝𝐨𝐫𝐞𝐬/𝐚𝐬 𝐏𝐇𝐏 𝐋𝐀𝐑𝐀𝐕𝐄𝐋 👩💻 👨...,"[{'link': 'http://www.proconsi.com/', 'text': ...","[1 day ago, Full-time, No degree mentioned]",eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIFBIUC...,1 days 00:00:00,Full-time,https://encrypted-tbn0.gstatic.com/images?q=tb...,NaN,NaN,2024-04-16,"[CSS, JavaScript, JSON, PHP, Javascript]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22189,Analista Programador Dynamics (CRM),X25 Informática S.L.,"Madrid, Spain",LinkedIn,Buscamos un Desarrollador Dynamics CRM para co...,[{'items': ['Buscamos un Desarrollador Dynamic...,[{'link': 'https://www.google.com/search?sca_e...,"[7 hours ago, Full-time, No degree mentioned]",eyJqb2JfdGl0bGUiOiJBbmFsaXN0YSBQcm9ncmFtYWRvci...,0 days 07:00:00,Full-time,https://encrypted-tbn0.gstatic.com/images?q=tb...,NaN,NaN,2024-04-29,"[SQL, CRM, Dynamics]"
22190,Científico/a de Datos,ASOCIACION PARA EL DESARROLLO DE LA INGENIERIA...,Spain,Glassdoor,¿Qué te pedimos?\n• Doble titulación en Ingeni...,[{'items': ['¿Qué te pedimos? • Doble titulaci...,[{'link': 'https://www.google.com/search?sca_e...,"[25 days ago, Full-time]",eyJqb2JfdGl0bGUiOiJDaWVudMOtZmljby9hIGRlIERhdG...,25 days 00:00:00,Full-time,NaN,NaN,NaN,2024-04-04,"[Aceptación, GitHub, DOS]"
22191,Informático a domicilio para configurar equipo...,Cronoshare,"San Lorenzo de El Escorial, Spain",BeBee,Necesito un servicio de Informático a domicili...,[{'items': ['Necesito un servicio de Informáti...,[{'link': 'https://www.google.com/search?sca_e...,"[4 days ago, Full-time, No degree mentioned]",eyJqb2JfdGl0bGUiOiJJbmZvcm3DoXRpY28gYSBkb21pY2...,4 days 00:00:00,Full-time,https://encrypted-tbn0.gstatic.com/images?q=tb...,NaN,NaN,2024-04-25,[]
22192,Inside Technician TCS,Diversey,"Viladecans, Spain",BeBee,Responsabilidades:\n\nResponsable del soporte ...,[{'items': ['Responsabilidades: Responsable d...,"[{'link': 'http://www.diversey.com/', 'text': ...","[4 days ago, Full-time]",eyJqb2JfdGl0bGU

In [9]:
preprocess.get_skills(df[df["title"] == "Científico/a de Datos"]["description"][22190])

['Aceptación', 'GitHub', 'DOS']

In [10]:
df.iloc[22190,:]

title                                                             Científico/a de Datos
company_name                          ASOCIACION PARA EL DESARROLLO DE LA INGENIERIA...
location                                                                          Spain
via                                                                           Glassdoor
description                           ¿Qué te pedimos?\n• Doble titulación en Ingeni...
job_highlights                        [{'items': ['¿Qué te pedimos?
• Doble titulaci...
related_links                         [{'link': 'https://www.google.com/search?sca_e...
extensions                                                     [25 days ago, Full-time]
job_id                                eyJqb2JfdGl0bGUiOiJDaWVudMOtZmljby9hIGRlIERhdG...
detected_extensions.posted_at                                          25 days 00:00:00
detected_extensions.schedule_type                                             Full-time
thumbnail                       

In [11]:
df["tech_skills"].value_counts()

tech_skills
[]                                                                                 5580
[Integración]                                                                       475
[SAP]                                                                               366
[DOS]                                                                               273
[Marketing]                                                                         232
                                                                                   ... 
[CSS, DB2, Funcionales, Integración, jQuery, Kanban, Oracle, Scrum, Sonar, SQL]       1
[Funcionales, Integración, Marketing]                                                 1
[Java, Twitter]                                                                       1
[Jira, Oracle, SQL, SAS, Training]                                                    1
[Azure, Prisma]                                                                       1
Name: count, Length:

In [12]:
def clean_column_names(df):
    
    mapper = {"via"                               : "source",
              "detected_extensions.schedule_type" : "contract_type"}
    
    df = df.rename(mapper = mapper, axis = 1)    
    return df

In [13]:
# Renombramos los nombres de las columnas
df = clean_column_names(df)
# Nos quedamos con las columnas que nos interesan
df = df[["job_id", "title", "company_name", "location", "source",
         "description", "date_posted", "contract_type", "tech_skills"]]

df.shape

(22194, 9)

In [14]:
df

,job_id,title,company_name,location,source,description,date_posted,contract_type,tech_skills
0,eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIFNvZn...,Desarrollador de Software,THE WISE SEEKER S.L,Spain,Infoempleo,- Desarrollar soluciones de software asegurand...,2024-04-17,Full-time,[]
1,eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIFNvZn...,Desarrollador de Software,WITRON Logistik + Informatik GmbH,"Madrid, Spain",Glassdoor,Als Teil der WITRON Gruppe plant und realisier...,2024-03-20,Full-time,[SQL]
2,eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIHNvZn...,Desarrollador de software,CBT,"Getxo, Spain",LinkedIn,CBT es una empresa del Grupo Innovalia dedicad...,2024-04-15,Full-time,"[Angular, Android, iPhone]"
3,eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yL2EgZGUgc2...,Desarrollador/a de software en C/C+,SEGULA Technologies,"Barcelona, Spain",LinkedIn,Localidad : Barcelona\n\nProvincia : Barcelona...,2024-04-15,Full-time,"[Jira, REST]"
4,eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIFBIUC...,Desarrollador de PHP,Proconsi,"León, Spain",LinkedIn,𝐏𝐫𝐨𝐠𝐫𝐚𝐦𝐚𝐝𝐨𝐫𝐞𝐬/𝐚𝐬 𝐏𝐇𝐏 𝐋𝐀𝐑𝐀𝐕𝐄𝐋 👩💻 👨💻\n\n¿𝐏𝐫𝐨𝐠𝐫𝐚𝐦...,2024-04-16,Full-time,"[CSS, JavaScript, JSON, PHP, Javascript]"
...,...,...,...,...,...,...,...,...,...
22189,eyJqb2JfdGl0bGUiOiJBbmFsaXN0YSBQcm9ncmFtYWRvci...,Analista Programador Dynamics (CRM),X25 Informática S.L.,"Madrid, Spain",LinkedIn,Buscamos un Desarrollador Dynamics CRM para co...,2024-04-29,Full-time,"[SQL, CRM, Dynamics]"
22190,eyJqb2JfdGl0bGUiOiJDaWVudMOtZmljby9hIGRlIERhdG...,Científico/a de Datos,ASOCIACION PARA EL DESARROLLO DE LA INGENIERIA...,Spain,Glassdoor,¿Qué te pedimos?\n• Doble titulación en Ingeni...,2024-04-04,Full-time,"[Aceptación, GitHub, DOS]"
22191,eyJqb2JfdGl0bGUiOiJJbmZvcm3DoXRpY28gYSBkb21pY2...,Informático a domicilio para configurar equipo...,Cronoshare,"San Lorenzo de El Escorial, Spain",BeBee,Necesito un servicio de Informático a domicili...,2024-04-25,Full-time,[]
22192,eyJqb2JfdGl0bGUiOiJJbnNpZGUgVGVjaG5pY2lhbiBUQ1...,Inside Technician TCS,Diversey,"Viladecans, Spain",BeBee,Responsabilidades:\n\nResponsable del soporte ...,2024-04-25,Full-time,"[ERP, Microsoft, SAP]"


In [15]:
# Funciones para generar las columnas de años de experiencia y nivel de experiencia
def find_years_of_experience(string: str):
    
    list_strings = ["años de", "years of", "years experience", "años experiencia"]
    
    string = string.lower()
    
    years = [string[string.find(s) - 5 : string.find(s) + len(s) + 1] for s in list_strings if string.find(s) != -1]
    
    numeros = [re.findall(r"\d+", y) for y in years]

    numeros = [[int(n) for n in num if 0 < int(n) < 13] for num in numeros]
    
    numeros = [max(num) if num else np.nan for num in numeros]

    return max(numeros) if numeros else np.nan


def experience_level(num):
    
    if not pd.isna(num):
    
        if num < 2:
            return "Junior"
        elif num <= 4:
            return "Semi-Senior"
        elif num < 8:
            return "Senior"
        else:
            return "Leader"
        
    else:
        return np.nan

In [16]:
df["experience"] = df["description"].apply(lambda x : find_years_of_experience(x) if not pd.isna(x) else x)
df["experience_level"] = df["experience"].apply(lambda x : experience_level(x))

In [17]:
# Actualizamos "contract_type"
df["contract_type"] = df["contract_type"]\
                            .apply(lambda x : "Full-time" if x == "Tiempo completo" else x)

# Creamos "remote_work"
def get_remote_work(string):
    
    resultados = list()
    
    if "remoto" in string or "remote work" in string or "remote" in string:
        
        resultados.append("Remoto")
        
    elif "hibrido" in string or "hybrid" in string or "híbrido" in string:
        
        resultados.append("Hibrido")
        
    elif "presencial" in string or "in-office" in string:
        
        resultados.append("Presencial")
        
    else:
        return np.nan
        
    return resultados

df["remote_work"] = df["description"].apply(lambda x : get_remote_work(x))

# Actualizamos "location"
# ¡PENDIENTE, CONSULTAR CON DANI!
# ACTUALIZADO en la librería de preprocesamiento para el pipeline

In [18]:
df

,job_id,title,company_name,location,source,description,date_posted,contract_type,tech_skills,experience,experience_level,remote_work
0,eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIFNvZn...,Desarrollador de Software,THE WISE SEEKER S.L,Spain,Infoempleo,- Desarrollar soluciones de software asegurand...,2024-04-17,Full-time,[],NaN,NaN,NaN
1,eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIFNvZn...,Desarrollador de Software,WITRON Logistik + Informatik GmbH,"Madrid, Spain",Glassdoor,Als Teil der WITRON Gruppe plant und realisier...,2024-03-20,Full-time,[SQL],NaN,NaN,NaN
2,eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIHNvZn...,Desarrollador de software,CBT,"Getxo, Spain",LinkedIn,CBT es una empresa del Grupo Innovalia dedicad...,2024-04-15,Full-time,"[Angular, Android, iPhone]",NaN,NaN,NaN
3,eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yL2EgZGUgc2...,Desarrollador/a de software en C/C+,SEGULA Technologies,"Barcelona, Spain",LinkedIn,Localidad : Barcelona\n\nProvincia : Barcelona...,2024-04-15,Full-time,"[Jira, REST]",3.0,Semi-Senior,NaN
4,eyJqb2JfdGl0bGUiOiJEZXNhcnJvbGxhZG9yIGRlIFBIUC...,Desarrollador de PHP,Proconsi,"León, Spain",LinkedIn,𝐏𝐫𝐨𝐠𝐫𝐚𝐦𝐚𝐝𝐨𝐫𝐞𝐬/𝐚𝐬 𝐏𝐇𝐏 𝐋𝐀𝐑𝐀𝐕𝐄𝐋 👩💻 👨💻\n\n¿𝐏𝐫𝐨𝐠𝐫𝐚𝐦...,2024-04-16,Full-time,"[CSS, JavaScript, JSON, PHP, Javascript]",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
22189,eyJqb2JfdGl0bGUiOiJBbmFsaXN0YSBQcm9ncmFtYWRvci...,Analista Programador Dynamics (CRM),X25 Informática S.L.,"Madrid, Spain",LinkedIn,Buscamos un Desarrollador Dynamics CRM para co...,2024-04-29,Full-time,"[SQL, CRM, Dynamics]",NaN,NaN,NaN
22190,eyJqb2JfdGl0bGUiOiJDaWVudMOtZmljby9hIGRlIERhdG...,Científico/a de Datos,ASOCIACION PARA EL DESARROLLO DE LA INGENIERIA...,Spain,Glassdoor,¿Qué te pedimos?\n• Doble titulación en Ingeni...,2024-04-04,Full-time,"[Aceptación, GitHub, DOS]",NaN,NaN,NaN
22191,eyJqb2JfdGl0bGUiOiJJbmZvcm3DoXRpY28gYSBkb21pY2...,Informático a domicilio para configurar equipo...,Cronoshare,"San Lorenzo de El Escorial, Spain",BeBee,Necesito un servicio de Informático a domicili...,2024-04-25,Full-time,[],NaN,NaN,[Remoto]
22192,eyJqb2JfdGl0bGUiOiJJbnNpZGUgVGVjaG5pY2lhbiBUQ1...,Inside Technician TCS,Diversey,"Viladecans, Spain",BeBee,Responsabilidades:\n\nResponsable del soporte ...,2024-04-25,Full-time,"[ERP, Microsoft, SAP]",NaN,NaN,[Remoto]


In [19]:
def clean_location_spain(x):
    
    if type(x) == str:
        
        if x == "Spain":
            return "Spain"
        
        elif x == "Anywhere":
            return "Anywhere"
        
        if "," in x:
            
            x = x.split(", ")[0]
            
            if x == "Community of Madrid":
                return "Madrid"
            
            return x
        
    return np.nan

In [20]:
df["location"] = df["location"].apply(lambda x : clean_location_spain(x))

In [21]:
df["location"].value_counts(dropna = False, normalize = True)

location
Madrid        0.317023
Anywhere      0.144499
Barcelona     0.106515
Spain         0.098991
Valencia      0.027260
                ...   
Canovelles    0.000045
Montilla      0.000045
La Zubia      0.000045
Santa Pola    0.000045
Monte         0.000045
Name: proportion, Length: 812, dtype: float64

In [22]:
def eliminar_caracteres_especiales(x):
    patron = r"[^\w\s]"
    return re.sub(patron, "", x)

# Empresas

In [24]:
df_company = pd.read_csv("empresas_v1.csv").dropna().reset_index(drop = True)

df_company["company_name_lower"] = df_company["company_name"].apply(lambda x : x.lower())
df_company["company_name_lower_no_spaces"] = df_company["company_name_lower"].apply(lambda x : x.replace(" ", ""))
df_company["company_name_lower_no_spaces_no_special_char"] = df_company["company_name_lower_no_spaces"].apply(lambda x : eliminar_caracteres_especiales(x))

df_company

,company_name,company_name_lower,company_name_lower_no_spaces,company_name_lower_no_spaces_no_special_char
0,THE WISE SEEKER S.L,the wise seeker s.l,thewiseseekers.l,thewiseseekersl
1,WITRON Logistik + Informatik GmbH,witron logistik + informatik gmbh,witronlogistik+informatikgmbh,witronlogistikinformatikgmbh
2,CBT,cbt,cbt,cbt
3,SEGULA Technologies,segula technologies,segulatechnologies,segulatechnologies
4,Proconsi,proconsi,proconsi,proconsi
...,...,...,...,...
4805,Came Parkare,came parkare,cameparkare,cameparkare
4806,PROYECSON,proyecson,proyecson,proyecson
4807,Acrelec,acrelec,acrelec,acrelec
4808,Ctrl360,ctrl360,ctrl360,ctrl360


In [25]:
def levenshtein_distance(str1, str2):

    d = dict()
    
    for i in range(len(str1)+1):
        
        d[i]=dict()
        d[i][0]=i
        
    for i in range(len(str2)+1):
        
        d[0][i] = i
        
    for i in range(1, len(str1)+1):
        for j in range(1, len(str2)+1):
            
            d[i][j] = min(d[i][j-1]+1, d[i-1][j]+1, d[i-1][j-1]+(not str1[i-1] == str2[j-1]))
            
            
    return d[len(str1)][len(str2)]

In [26]:
# Sacamos las compañias del df de extracción y transformamos a minusculas sin espacios y sin caracteres especiales

companies = list({eliminar_caracteres_especiales(x.lower().replace(" ", "")) for x in df["company_name"].unique()})

len(companies)

6289

In [27]:
companies

['telefónicatechgovertis',
 'ilunion',
 'turnertownsend',
 'ctrl360',
 'abalia',
 'riekatalentett',
 'aimhiregloballtd',
 'airtificialaerospacedefense',
 'enerlandgroup',
 'centricsoftware',
 'trentiaconsulting',
 'ednon',
 'cumminscareers',
 'hdivsecurity',
 'rocasalvatella',
 'cwelltsoftwaresl',
 'sesameuniverse_',
 'gruporetabet',
 'invenergy',
 'arvatosupplychainsolutions',
 'inetum',
 'vatesanepamcompany',
 'openassessmenttechnologiessa',
 'zitro',
 'aszendittech',
 'helphone',
 'talenthunter',
 'utopiux',
 'anecoopscoop',
 'magtel',
 'grupoamper',
 'atentoespaña',
 'grupoespiral',
 'abbschweizag',
 'clienterandstad',
 'capitalempresarialhorizonte',
 'metyis',
 'trentia',
 'gruposurmaster',
 'techstaq',
 'coresystemsserviciosysoluciones',
 'enrique',
 'cornerjobstaffing',
 'sermica',
 'universae',
 'grupovolmae',
 'acquirehrsolutions',
 'cunovesagroup',
 'artworkscomunicación',
 'alejandra',
 'academiageorgetown',
 'esmlsdiberiaholdingsau',
 'dimaimsystemssl',
 'panelsistemasinfor

In [28]:
for x in df_company["company_name_lower_no_spaces_no_special_char"]:
    if x == "kenvue":
        print(x)
        break

kenvue


In [29]:
df_company[df_company["company_name_lower_no_spaces_no_special_char"] == "kenvue"]

,company_name,company_name_lower,company_name_lower_no_spaces,company_name_lower_no_spaces_no_special_char
4585,Kenvue,kenvue,kenvue,kenvue


In [30]:
# Verificamos si en "companies" existen ya algunas de df_company

inter_companies = [x for x in companies if x in df_company["company_name_lower_no_spaces_no_special_char"].values]

print(f"Tamaño de 'companies:' {len(companies)}")
print(f"Tamaño de interseccion: {len(inter_companies)}")





Tamaño de 'companies:' 6289
Tamaño de interseccion: 4510


In [31]:
np.array(df["location"].value_counts(dropna = False).index)

array(['Madrid', 'Anywhere', 'Barcelona', 'Spain', 'Valencia', 'Seville',
       'Málaga', 'Bilbao', 'Zaragoza', 'Murcia', 'Tres Cantos',
       'Alicante', 'A Coruña', 'Valladolid', 'Granada', 'Alcobendas',
       'Córdoba', 'Pozuelo de Alarcón', 'Sant Cugat del Vallès',
       'Las Palmas de Gran Canaria', 'Palma', 'Pamplona', 'Vigo',
       'Tarragona', 'Girona', 'Donostia-San Sebastian', 'Catalonia',
       'Getafe', 'Santa Cruz de Tenerife', 'Oviedo', 'Basque Country',
       'Las Rozas de Madrid', 'Albacete', 'Vitoria-Gasteiz', 'Andalusia',
       'Torrejón de Ardoz', 'Gijón', 'Santiago de Compostela', 'León',
       'Paterna', 'Almería', 'Salamanca', 'Asturias', 'Santander',
       'Burgos', 'Castellón de la Plana', 'Lleida',
       'San Sebastián de los Reyes', 'Boadilla del Monte', 'Biscay',
       'Terrassa', 'Cádiz', 'Toledo', 'Martorell',
       "L'Hospitalet de Llobregat", 'Alcala de Henares',
       'San Fernando de Henares', 'Elche', 'Jaén', 'Pontevedra',
       'Ciudad 

In [32]:
df.fillna(np.nan).replace([np.nan], [None]).to_csv("bigquery_1.csv", index = False)

In [33]:
df.to_csv("bigquery_1.csv", index = False)